# **1. Mounting Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **2. Install Dependencies for PySpark**

Installed Java, PySpark, and findspark to set up a working Spark environment in this new Colab session.

In [2]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q -U "pyspark[connect]~=4.0.0" findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# **3. SparkSession Initialisation**

I initialised the SparkSession with 8GB of memory, since fitting four models with CrossValidator needs sufficient resources to run smoothly.

In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark.sql.types as T

spark = (
    SparkSession.builder
    .appName("AmazonAutomotiveModelTraining")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version : {spark.version}")

Spark version : 4.0.4


# **4. Loading Processed Data from Parquet File**

I loaded the training and test Parquet files saved in the preprocessing notebook to confirm row counts and schema before proceeding to model training.

In [4]:
output_dir = "/content/drive/MyDrive/amazon_automotive_sentiment"

training_data = spark.read.parquet(f"{output_dir}/training_data.parquet")
testing_data  = spark.read.parquet(f"{output_dir}/testing_data.parquet")

print(f"Training rows : {training_data.count():,}")
print(f"Testing rows  : {testing_data.count():,}")
training_data.printSchema()

Training rows : 15,763,810
Testing rows  : 3,943,422
root
 |-- sentiment: double (nullable = true)
 |-- features: vector (nullable = true)



# **5. Sampling Training and Test Data**

I sampled the data down to keep CrossValidator's training time manageable, since fitting multiple models per algorithm on millions of rows would take many hours per model.

In [5]:
training_sample = training_data.sample(fraction=0.002, seed=42)
testing_sample  = testing_data.sample(fraction=0.004, seed=42)

print(f"Sampled training rows : {training_sample.count():,}")
print(f"Sampled testing rows  : {testing_sample.count():,}")

Sampled training rows : 31,551
Sampled testing rows  : 15,788


# **6. Setting Up Evaluation Metrics**

I defined five evaluators to assess each model from multiple angles: AUC to measure how well each model separates Positive from Negative reviews, alongside accuracy, precision, recall, and F1-score.

I used the weighted variants of precision and recall because the dataset is imbalanced, with roughly 78% Positive reviews, so these metrics may reflect performance across both classes rather than being skewed toward the majority class.


In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import time

target = "sentiment"

auc_eval       = BinaryClassificationEvaluator(labelCol=target, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
accuracy_eval  = MulticlassClassificationEvaluator(labelCol=target, metricName="accuracy")
precision_eval = MulticlassClassificationEvaluator(labelCol=target, metricName="weightedPrecision")
recall_eval    = MulticlassClassificationEvaluator(labelCol=target, metricName="weightedRecall")
f1_eval        = MulticlassClassificationEvaluator(labelCol=target, metricName="f1")

model_results = {}
models_dir = f"{output_dir}/models"

# **7. Model 1 - Logistic Regression**

**Model and Grid Setup**

I started with Logistic Regression because review sentiment is often driven by a few strong signal words ("great", "terrible", "broken") that push the prediction clearly toward Positive or Negative.

A linear model captures this kind of pattern well, and it's also the fastest of the four algorithms to train which makes it a sensible baseline to compare the other models against it. I tuned regParam to control how much the model penalises large coefficients. It also helps the model generalise better to reviews it hasn't seen before, rather than memorising patterns specific to the training data.

In [7]:
from pyspark.ml.classification import LogisticRegression

logit = LogisticRegression(labelCol=target, featuresCol="features", maxIter=10)

logit_grid = (ParamGridBuilder()
              .addGrid(logit.regParam, [0.01, 0.1])
              .build())

logit_cv = CrossValidator(
    estimator=Pipeline(stages=[logit]), estimatorParamMaps=logit_grid,
    evaluator=auc_eval, numFolds=2, parallelism=1, seed=42
)

print(f"Logistic Regression: {len(logit_grid)} combinations x 2 folds")

Logistic Regression: 2 combinations x 2 folds


# **8. Logistic Regression - Model Training**

In [8]:
logit_start = time.time()
logit_fitted = logit_cv.fit(training_sample)
logit_time = round(time.time() - logit_start, 2)

print(f"Training complete : {logit_time} seconds ({logit_time/60:.1f} minutes)")

Training complete : 251.64 seconds (4.2 minutes)


# **9. Logistic Regression - Model Evaluation and Saving**

In [9]:
logit_preds = logit_fitted.bestModel.transform(testing_sample)

logit_auc  = auc_eval.evaluate(logit_preds)
logit_acc  = accuracy_eval.evaluate(logit_preds)
logit_prec = precision_eval.evaluate(logit_preds)
logit_rec  = recall_eval.evaluate(logit_preds)
logit_f1   = f1_eval.evaluate(logit_preds)

best_logit = logit_fitted.bestModel.stages[-1]

print("LOGISTIC REGRESSION RESULTS")
print("=" * 32)
print(f"{'Metric':<18}{'Value'}")
print("-" * 32)
print(f"{'AUC':<18}{logit_auc:.4f}")
print(f"{'Accuracy':<18}{logit_acc:.4f}")
print(f"{'Precision':<18}{logit_prec:.4f}")
print(f"{'Recall':<18}{logit_rec:.4f}")
print(f"{'F1 Score':<18}{logit_f1:.4f}")
print("-" * 32)
print(f"{'Training time':<18}{logit_time:.2f}s")
print(f"{'Best regParam':<18}{best_logit._java_obj.getRegParam()}")
print("=" * 32)

model_results["Logistic Regression"] = {
    "time": logit_time, "auc": logit_auc, "acc": logit_acc,
    "precision": logit_prec, "recall": logit_rec, "f1": logit_f1
}

logit_fitted.bestModel.write().overwrite().save(f"{models_dir}/logistic_regression")
print("Model saved.")

LOGISTIC REGRESSION RESULTS
Metric            Value
--------------------------------
AUC               0.9200
Accuracy          0.8697
Precision         0.8644
Recall            0.8697
F1 Score          0.8576
--------------------------------
Training time     251.64s
Best regParam     0.1
Model saved.


# **10. Model 2 - Random Forest**

**Model and Grid Setup**

I used Random Forest as my second model since it builds on Decision Tree's logic but trains many trees on different random subsets of the data, then averages their predictions. This reduces the risk of any single tree overfitting to noise in the reviews. I tuned numTrees to find the right balance between accuracy and training time, and kept maxDepth shallow with conservative memory settings, since the 65,541-dimension TF-IDF vectors make each tree expensive to build.



In [10]:
from pyspark.ml.classification import RandomForestClassifier

forest = RandomForestClassifier(
    labelCol=target, featuresCol="features", seed=42,
    maxDepth=3, maxBins=32, maxMemoryInMB=512, subsamplingRate=0.8
)

forest_grid = (ParamGridBuilder()
               .addGrid(forest.numTrees, [5, 10])
               .build())

forest_cv = CrossValidator(
    estimator=Pipeline(stages=[forest]), estimatorParamMaps=forest_grid,
    evaluator=auc_eval, numFolds=2, parallelism=1, seed=42
)

print(f"Random Forest: {len(forest_grid)} combinations x 2 folds")

Random Forest: 2 combinations x 2 folds


# **11. Random Forest — Model Training**

In [11]:
forest_start = time.time()
forest_fitted = forest_cv.fit(training_sample)
forest_time = round(time.time() - forest_start, 2)

print(f"Training complete : {forest_time} seconds ({forest_time/60:.1f} minutes)")

Training complete : 694.62 seconds (11.6 minutes)


# **12. Random Forest — Model Evaluation and Saving**

In [12]:
forest_preds = forest_fitted.bestModel.transform(testing_sample)

forest_auc  = auc_eval.evaluate(forest_preds)
forest_acc  = accuracy_eval.evaluate(forest_preds)
forest_prec = precision_eval.evaluate(forest_preds)
forest_rec  = recall_eval.evaluate(forest_preds)
forest_f1   = f1_eval.evaluate(forest_preds)

best_forest = forest_fitted.bestModel.stages[-1]

print("RANDOM FOREST RESULTS")
print("=" * 32)
print(f"{'Metric':<18}{'Value'}")
print("-" * 32)
print(f"{'AUC':<18}{forest_auc:.4f}")
print(f"{'Accuracy':<18}{forest_acc:.4f}")
print(f"{'Precision':<18}{forest_prec:.4f}")
print(f"{'Recall':<18}{forest_rec:.4f}")
print(f"{'F1 Score':<18}{forest_f1:.4f}")
print("-" * 32)
print(f"{'Training time':<18}{forest_time:.2f}s")
print(f"{'Best numTrees':<18}{best_forest.getNumTrees}")
print("=" * 32)

model_results["Random Forest"] = {
    "time": forest_time, "auc": forest_auc, "acc": forest_acc,
    "precision": forest_prec, "recall": forest_rec, "f1": forest_f1
}

forest_fitted.bestModel.write().overwrite().save(f"{models_dir}/random_forest")
print("Model saved.")

RANDOM FOREST RESULTS
Metric            Value
--------------------------------
AUC               0.5705
Accuracy          0.7875
Precision         0.6202
Recall            0.7875
F1 Score          0.6939
--------------------------------
Training time     694.62s
Best numTrees     10
Model saved.


# **13. Model 3 - Gradient Boosted Trees**

**Model and Grid Setup**


I included Gradient Boosted Trees as my third model as unlike Random Forest which builds trees independently, GBT builds trees one after another and each new tree focuses on correcting the mistakes of the previous ones.

This sequential learning often captures subtler patterns in the data that the other models miss. I tuned stepSize to control how aggressively each new tree corrects errors, and kept the trees shallow with few iterations, since the high-dimensional TF-IDF features make each boosting round computationally expensive.

In [13]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    labelCol=target, featuresCol="features", seed=42,
    maxDepth=2, maxIter=5, maxBins=32
)

gbt_grid = (ParamGridBuilder()
            .addGrid(gbt.stepSize, [0.1, 0.2])
            .build())

gbt_cv = CrossValidator(
    estimator=Pipeline(stages=[gbt]), estimatorParamMaps=gbt_grid,
    evaluator=auc_eval, numFolds=2, parallelism=1, seed=42
)

print(f"GBT: {len(gbt_grid)} combinations x 2 folds")

GBT: 2 combinations x 2 folds


# **14. Gradient Boosted Trees — Model Training**

In [14]:
gbt_start = time.time()
gbt_fitted = gbt_cv.fit(training_sample)
gbt_time = round(time.time() - gbt_start, 2)

print(f"Training complete : {gbt_time} seconds ({gbt_time/60:.1f} minutes)")

Training complete : 984.9 seconds (16.4 minutes)


# **15. Gradient Boosted Trees — Model Evaluation and Saving**

In [15]:
gbt_preds = gbt_fitted.bestModel.transform(testing_sample)

gbt_auc  = auc_eval.evaluate(gbt_preds)
gbt_acc  = accuracy_eval.evaluate(gbt_preds)
gbt_prec = precision_eval.evaluate(gbt_preds)
gbt_rec  = recall_eval.evaluate(gbt_preds)
gbt_f1   = f1_eval.evaluate(gbt_preds)

best_gbt = gbt_fitted.bestModel.stages[-1]

print("GRADIENT BOOSTED TREES RESULTS")
print("=" * 32)
print(f"{'Metric':<18}{'Value'}")
print("-" * 32)
print(f"{'AUC':<18}{gbt_auc:.4f}")
print(f"{'Accuracy':<18}{gbt_acc:.4f}")
print(f"{'Precision':<18}{gbt_prec:.4f}")
print(f"{'Recall':<18}{gbt_rec:.4f}")
print(f"{'F1 Score':<18}{gbt_f1:.4f}")
print("-" * 32)
print(f"{'Training time':<18}{gbt_time:.2f}s")
print(f"{'Best stepSize':<18}{best_gbt.getStepSize()}")
print("=" * 32)

model_results["Gradient Boosted Trees"] = {
    "time": gbt_time, "auc": gbt_auc, "acc": gbt_acc,
    "precision": gbt_prec, "recall": gbt_rec, "f1": gbt_f1
}

gbt_fitted.bestModel.write().overwrite().save(f"{models_dir}/gbt")
print("Model saved.")

GRADIENT BOOSTED TREES RESULTS
Metric            Value
--------------------------------
AUC               0.7831
Accuracy          0.8113
Precision         0.8270
Recall            0.8113
F1 Score          0.7502
--------------------------------
Training time     984.90s
Best stepSize     0.2
Model saved.


# **16. Model 4 - Naive Bayes**

**Model and Grid Setup**

I included Naive Bayes as my fourth model since it's mainly designed for text classification. It calculates word probabilities directly from the TF-IDF features rather than relying on linear boundaries or tree splits like the other three models. This makes it a useful comparison point, even though its assumption that words are independent of each other is a simplification that doesn't always hold true for natural language. I tuned smoothing to prevent the model from assigning zero probability to words it hasn't seen in training.

In [16]:
from pyspark.ml.classification import NaiveBayes

nbayes = NaiveBayes(labelCol=target, featuresCol="features", modelType="multinomial")

nbayes_grid = (ParamGridBuilder()
               .addGrid(nbayes.smoothing, [0.5, 1.0])
               .build())

nbayes_cv = CrossValidator(
    estimator=Pipeline(stages=[nbayes]), estimatorParamMaps=nbayes_grid,
    evaluator=auc_eval, numFolds=2, parallelism=1, seed=42
)

print(f"Naive Bayes: {len(nbayes_grid)} combinations x 2 folds")

Naive Bayes: 2 combinations x 2 folds


# **17. Naive Bayes — Model Training**

In [17]:
nbayes_start = time.time()
nbayes_fitted = nbayes_cv.fit(training_sample)
nbayes_time = round(time.time() - nbayes_start, 2)

print(f"Training complete : {nbayes_time} seconds ({nbayes_time/60:.1f} minutes)")

Training complete : 156.2 seconds (2.6 minutes)


# **18. Naive Bayes — Model Evaluation and Saving**

In [18]:
nbayes_preds = nbayes_fitted.bestModel.transform(testing_sample)

nbayes_auc  = auc_eval.evaluate(nbayes_preds)
nbayes_acc  = accuracy_eval.evaluate(nbayes_preds)
nbayes_prec = precision_eval.evaluate(nbayes_preds)
nbayes_rec  = recall_eval.evaluate(nbayes_preds)
nbayes_f1   = f1_eval.evaluate(nbayes_preds)

best_nbayes = nbayes_fitted.bestModel.stages[-1]

print("NAIVE BAYES RESULTS")
print("=" * 32)
print(f"{'Metric':<18}{'Value'}")
print("-" * 32)
print(f"{'AUC':<18}{nbayes_auc:.4f}")
print(f"{'Accuracy':<18}{nbayes_acc:.4f}")
print(f"{'Precision':<18}{nbayes_prec:.4f}")
print(f"{'Recall':<18}{nbayes_rec:.4f}")
print(f"{'F1 Score':<18}{nbayes_f1:.4f}")
print("-" * 32)
print(f"{'Training time':<18}{nbayes_time:.2f}s")
print(f"{'Best smoothing':<18}{best_nbayes.getSmoothing()}")
print("=" * 32)

model_results["Naive Bayes"] = {
    "time": nbayes_time, "auc": nbayes_auc, "acc": nbayes_acc,
    "precision": nbayes_prec, "recall": nbayes_rec, "f1": nbayes_f1
}

nbayes_fitted.bestModel.write().overwrite().save(f"{models_dir}/naive_bayes")
print("Model saved.")

NAIVE BAYES RESULTS
Metric            Value
--------------------------------
AUC               0.6407
Accuracy          0.7579
Precision         0.8097
Recall            0.7579
F1 Score          0.7741
--------------------------------
Training time     156.20s
Best smoothing    1.0
Model saved.


# **19. Final Model Comparison Table**

In [19]:
comparison_data = [
    (name, stats["time"], stats["acc"], stats["precision"], stats["recall"], stats["f1"], stats["auc"])
    for name, stats in model_results.items()
]

comparison_schema = T.StructType([
    T.StructField("Algorithm",       T.StringType(), False),
    T.StructField("Training_Time_s", T.DoubleType(), False),
    T.StructField("Accuracy",        T.DoubleType(), False),
    T.StructField("Precision",       T.DoubleType(), False),
    T.StructField("Recall",          T.DoubleType(), False),
    T.StructField("F1_Score",        T.DoubleType(), False),
    T.StructField("AUC",             T.DoubleType(), False),
])

comparison_df = spark.createDataFrame(comparison_data, comparison_schema)

print("=" * 80)
print("FINAL MODEL COMPARISON TABLE")
print("=" * 80)

comparison_df.select(
    "Algorithm",
    F.round("Training_Time_s", 1).alias("Time_s"),
    F.round("Accuracy", 4).alias("Accuracy"),
    F.round("Precision", 4).alias("Precision"),
    F.round("Recall", 4).alias("Recall"),
    F.round("F1_Score", 4).alias("F1_Score"),
    F.round("AUC", 4).alias("AUC")
).show(truncate=False)

FINAL MODEL COMPARISON TABLE
+----------------------+------+--------+---------+------+--------+------+
|Algorithm             |Time_s|Accuracy|Precision|Recall|F1_Score|AUC   |
+----------------------+------+--------+---------+------+--------+------+
|Logistic Regression   |251.6 |0.8697  |0.8644   |0.8697|0.8576  |0.92  |
|Random Forest         |694.6 |0.7875  |0.6202   |0.7875|0.6939  |0.5705|
|Gradient Boosted Trees|984.9 |0.8113  |0.827    |0.8113|0.7502  |0.7831|
|Naive Bayes           |156.2 |0.7579  |0.8097   |0.7579|0.7741  |0.6407|
+----------------------+------+--------+---------+------+--------+------+



# **20. Saving Model Comparison Summary**

In [20]:
comparison_df.write.csv(f"{output_dir}/model_summary", header=True, mode="overwrite")
print(f"Summary saved to {output_dir}/model_summary")

Summary saved to /content/drive/MyDrive/amazon_automotive_sentiment/model_summary


# **21. Spark Session Termination**

In [21]:
spark.stop()
print("Spark session stopped successfully.")

Spark session stopped successfully.
